# 2. Geneformer Cell-Level Embedding Extraction

Извлечение cell-level mean-pool эмбеддингов из Geneformer foundation model.

**Архитектура Geneformer:**
```
Raw counts → rank-value encoding → TranscriptomeTokenizer
           → Geneformer Transformer (V2, 104M) → EmbExtractor (mean_pool)
                                                  ↑ WE EXTRACT HERE
```

Модель: `Geneformer-V2-104M_CLcancer` — fine-tuned на pan-cancer cell classification.

## 0. Environment Setup

In [ ]:
# !rm -rf batchcor-rna-embeds/  # if already cloned
!git clone https://github.com/onion-42/batchcor-rna-embeds.git
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q --upgrade ipython ipykernel

# Geneformer-specific deps
!pip install -q peft==0.10.0
!pip install -q transformers"<4.40.0"
!pip install -q anndata==0.10.9 scanpy==1.10.1
!pip install -q git+https://huggingface.co/ctheodoris/Geneformer.git

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import torch
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL,
    BATCH_SIZE,
    DIAGNOSIS_COL,
    EMB_LABEL_COLS,
    EMBS_LAYER,
    GENEFORMER_EMBEDDING_KEY,
    GENEFORMER_METADATA_KEY,
    GENEFORMER_MODEL_NAME,
    GENEFORMER_MODEL_URL,
    GENEFORMER_PCA_KEY,
    GENEFORMER_UMAP_KEY,
    MAX_CELLS,
    NPROC,
    SEED,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.geneformer_embedder import run_geneformer_pipeline

set_logger(level="INFO")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Device: {}", DEVICE)
logger.info("PyTorch: {}, CUDA available: {}", torch.__version__, torch.cuda.is_available())

## 1. Mount Drive & Define Paths

**ВАЖНО:** Исходные данные читаются из `data/raw`, результаты сохраняются в `data/interim`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_RAW_DIR     = Path('/content/drive/MyDrive/data/raw')
DATA_INTERIM_DIR = Path('/content/drive/MyDrive/data/interim')
DATA_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path('/content/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

logger.info("DATA_RAW_DIR:     {}", DATA_RAW_DIR)
logger.info("DATA_INTERIM_DIR: {}", DATA_INTERIM_DIR)
logger.info(
    "Available cohorts (raw): {}",
    [p.name for p in sorted(DATA_RAW_DIR.glob('*.adata.zarr'))],
)

## 2. Download Geneformer Model

Клонируем репозиторий модели через `git lfs` (веса ~400 MB).  
Если папка уже существует — пропускаем.

In [ ]:
MODEL_PATH = MODEL_DIR / GENEFORMER_MODEL_NAME

if not MODEL_PATH.exists():
    logger.info("Downloading Geneformer model from {}", GENEFORMER_MODEL_URL)
    !git lfs install
    !git clone {GENEFORMER_MODEL_URL} {MODEL_PATH}
    logger.info("Model cloned to: {}", MODEL_PATH)
else:
    logger.info("Model already exists: {}", MODEL_PATH)

# Verify expected subdirectory exists
target_model_dir = MODEL_PATH / GENEFORMER_MODEL_NAME
if target_model_dir.exists():
    MODEL_PATH = target_model_dir
    logger.info("Using model subdirectory: {}", MODEL_PATH)
else:
    logger.info("Using model directory directly: {}", MODEL_PATH)

## 3. Process All Cohorts

Для каждой когорты:
1. Load AnnData from `data/raw` Zarr
2. Tokenize (HUGO → Ensembl, rank-value encoding)
3. Extract Geneformer cell-level mean-pool embeddings
4. Compute PCA-128D + UMAP-3D
5. Store metadata in `.uns`
6. Save to `data/interim`

> **Gene mapping note:** `run_geneformer_pipeline` expects `adata.var_names` to be
> Ensembl IDs. If your raw data uses HUGO symbols, run
> `rename_adata_vars_to_ensembl` + `prepare_pseudo_counts` first (see cell below).

In [ ]:
#--- Optional: gene-symbol → Ensembl mapping (skip if already Ensembl) ---
# from batchcor_rna_emb.feature_calc.gene_mapping_geneformer import (
#     prepare_pseudo_counts,
#     rename_adata_vars_to_ensembl,
# )

# adata, found_genes, missing_genes = rename_adata_vars_to_ensembl(adata)
# logger.info(
#     "After mapping: {} genes retained, {} dropped",
#     len(found_genes), len(missing_genes),
# )
# adata = prepare_pseudo_counts(adata)

In [ ]:
# Cohorts to process — all raw Zarr stores found in DATA_RAW_DIR
cohort_paths = sorted(DATA_RAW_DIR.glob('*.adata.zarr'))
logger.info("Found {} cohort(s) to process.", len(cohort_paths))

for zarr_path in cohort_paths:
    cohort_name  = zarr_path.stem.replace('.adata', '')
    interim_path = DATA_INTERIM_DIR / zarr_path.name

    if interim_path.exists():
        logger.info("Already in interim, skipping: {}", cohort_name)
        continue

    logger.info("\n{'=' * 70}")
    logger.info("Processing cohort: {}", cohort_name)
    logger.info("{'=' * 70}")

    # Load from raw
    adata = load_cohort(zarr_path)
    
    from batchcor_rna_emb.feature_calc.gene_mapping_geneformer import (
    prepare_pseudo_counts,
    rename_adata_vars_to_ensembl,
    )

    adata, found_genes, missing_genes = rename_adata_vars_to_ensembl(adata)
    logger.info(
        "After mapping: {} genes retained, {} dropped",
        len(found_genes), len(missing_genes),
    )

    adata = prepare_pseudo_counts(adata)
    logger.info("Loaded: {} samples × {} genes", adata.n_obs, adata.n_vars)
    logger.info("Batch distribution:\n{}", adata.obs[BATCH_COL].value_counts())
    logger.info("Diagnosis distribution:\n{}", adata.obs[DIAGNOSIS_COL].value_counts())

    # Run Geneformer pipeline
    adata = run_geneformer_pipeline(
        adata,
        model_path=MODEL_PATH,
        emb_label_cols=EMB_LABEL_COLS,
        embs_layer=EMBS_LAYER,
        batch_size=BATCH_SIZE,
        max_cells=MAX_CELLS,
        nproc=NPROC,
        device=DEVICE,
        seed=SEED,
    )

    # Verify
    logger.info("Verification:")
    logger.info(
        "  .obsm['{}'] shape: {}, dtype: {}",
        GENEFORMER_EMBEDDING_KEY,
        adata.obsm[GENEFORMER_EMBEDDING_KEY].shape,
        adata.obsm[GENEFORMER_EMBEDDING_KEY].dtype,
    )
    logger.info(
        "  .obsm['{}'] shape: {}",
        GENEFORMER_PCA_KEY, adata.obsm[GENEFORMER_PCA_KEY].shape,
    )
    logger.info(
        "  .obsm['{}'] shape: {}",
        GENEFORMER_UMAP_KEY, adata.obsm[GENEFORMER_UMAP_KEY].shape,
    )
    logger.info(
        "  .uns['{}']: {}",
        GENEFORMER_METADATA_KEY, adata.uns[GENEFORMER_METADATA_KEY],
    )

    # Save to INTERIM (not raw!)
    save_adata_zarr(adata, interim_path)
    logger.info("Saved to interim: {}", interim_path)

    # Free memory
    del adata
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

logger.info("\nAll cohorts processed successfully.")
logger.info(
    "Interim contents: {}",
    [p.name for p in sorted(DATA_INTERIM_DIR.glob('*.adata.zarr'))],
)

## 4. Verification — Single Cohort Spot Check

In [ ]:
# Pick the first available cohort in interim for a quick sanity check
interim_stores = sorted(DATA_INTERIM_DIR.glob('*.adata.zarr'))

if not interim_stores:
    logger.warning("No cohorts found in interim — run Section 3 first.")
else:
    test_path = interim_stores[0]
    test_cohort = test_path.stem.replace('.adata', '')
    logger.info("Spot-checking cohort: '{}'", test_cohort)

    adata_check = load_cohort(test_path)

    assert GENEFORMER_EMBEDDING_KEY in adata_check.obsm, "Missing embedding key!"
    assert GENEFORMER_PCA_KEY       in adata_check.obsm, "Missing PCA key!"
    assert GENEFORMER_UMAP_KEY      in adata_check.obsm, "Missing UMAP key!"
    assert GENEFORMER_METADATA_KEY  in adata_check.uns,  "Missing metadata key!"

    emb = adata_check.obsm[GENEFORMER_EMBEDDING_KEY]
    assert emb.dtype == np.float32,           f"Wrong dtype: {emb.dtype}"
    assert emb.shape[0] == adata_check.n_obs, "Row count mismatch!"

    logger.info("All assertions passed for '{}'.", test_cohort)
    logger.info("  Embedding shape : {}", emb.shape)
    logger.info("  PCA shape       : {}", adata_check.obsm[GENEFORMER_PCA_KEY].shape)
    logger.info("  UMAP shape      : {}", adata_check.obsm[GENEFORMER_UMAP_KEY].shape)
    logger.info("  Metadata        : {}", adata_check.uns[GENEFORMER_METADATA_KEY])